# Tratativas

### Imports:

In [ ]:
#%pip install pandas
#%pip install openpyxl

In [ ]:
import pandas as pd
import re

### Criação do banco de dados .csv

In [ ]:
nome_arquivo_excel = '../dados/base_quedas_AACD_anonimizada.xlsx'
nome_arquivo_csv = '../dados/base_quedas_AACD_anonimizada.csv'

try:
    # Carregar as duas abas do Excel
    print("Lendo as abas do Excel...")
    df_60_mais = pd.read_excel(nome_arquivo_excel, sheet_name='60+')
    df_60_menos = pd.read_excel(nome_arquivo_excel, sheet_name='60-')

    # Função para juntar as linhas de 2 em 2
    def consertar_linhas_duplas(df):
        # Agrupa de 2 em 2 linhas e pega o primeiro valor não nulo
        df_agrupado = df.groupby(df.index // 2).first()
        return df_agrupado

    # Aplicar a correção nas duas tabelas
    print("Corrigindo o cadastro feito em duas linhas...")
    df_60_mais_corrigido = consertar_linhas_duplas(df_60_mais)
    df_60_menos_corrigido = consertar_linhas_duplas(df_60_menos)

    # Juntar as duas tabelas em uma só
    print("Unindo as duas tabelas...")
    df_final = pd.concat([df_60_mais_corrigido, df_60_menos_corrigido], ignore_index=True)

    # ---------------------------------------------------------
    print("Corrigindo a numeração dos IDs...")
    df_final['ID'] = range(1, len(df_final) + 1)
    # ---------------------------------------------------------

    # Salvar o resultado no arquivo .csv
    df_final.to_csv(nome_arquivo_csv, index=False, encoding='utf-8-sig')
    
    print(f"Sucesso! Arquivo '{nome_arquivo_csv}' criado com IDs sequenciais.")

except FileNotFoundError:
    print(f"Erro: O arquivo '{nome_arquivo_excel}' não foi encontrado na pasta.")
except Exception as e:
    print(f"Ocorreu um erro: {e}")

### Correções do nome das colunas:

In [ ]:
df = pd.read_csv('../dados/base_quedas_AACD_anonimizada.csv')

colunas_com_interrogacao = [
    'Clínica/Dx',
    'Aditamento 10m',
    'Status funcional',
    'Diagnóstico Funcional',
]
for coluna in colunas_com_interrogacao:
    df[coluna] = df[coluna].astype('string').str.replace('?', '', regex=False)

colunas = df.columns.tolist()
print(colunas)

In [ ]:
df.rename(columns={'Polifarmácia?': 'Polifarmácia'}, inplace=True)
print(df.columns)

In [ ]:
df.rename(columns={'Idade ': 'Idade'}, inplace=True)
print(df.columns)

### Correções nas colunas

- Apaga qualquer espaço em branco antes ou depois da palavra, transforma 'sim ' em 'sim'.

In [ ]:
df['Polifarmácia'] = (
    df['Polifarmácia']
    .astype('string')
    .str.strip()
    .replace({'': pd.NA})
    .fillna('não')
    .astype('category')
)

print(df['Polifarmácia'].dtype)
print(df['Polifarmácia'].value_counts())

- Remove a marca "?" antes da conversão numérica. Idades inválidas permanecem ausentes, sem imputação.

In [ ]:
df['Idade'] = df['Idade'].astype('string').str.replace('?', '', regex=False).str.strip()

In [ ]:
df['Idade'] = pd.to_numeric(df['Idade'], errors='coerce').astype('Int64')
# Calcula a média das idades (desconsiderando valores nulos)
df['Idade'] = pd.to_numeric(df['Idade'], errors='coerce').astype('Int64')
media_idade = df['Idade'].mean()
media_idade = int(round(media_idade))

df.loc[df['ID'] == 275, 'Idade'] = media_idade

#### Correções de tipagem

In [ ]:
df.tail(1)

In [ ]:
df.dtypes

In [ ]:
df = df.rename(columns={'Cognição         (10-CS)': 'Cogniçao (10-CS)'})

In [ ]:
df['Reação (blazepod)'] = pd.to_numeric(df['Reação (blazepod)'], errors='coerce').astype('Int64')
df['Reação (blazepod) DT'] = pd.to_numeric(df['Reação (blazepod) DT'], errors='coerce').astype('Int64')
df['CIF equilíbrio'] = pd.to_numeric(df['CIF equilíbrio'], errors='coerce').astype('Int64')
df['Ponte'] = pd.to_numeric(df['Ponte'], errors='coerce').astype('Int64')
df['Sentar/levantar 30s'] = pd.to_numeric(df['Sentar/levantar 30s'], errors='coerce').astype('Int64')
df['Nº quedas ao chão'] = pd.to_numeric(df['Nº quedas ao chão'], errors='coerce').astype('Int64')
df['Cogniçao (10-CS)'] = pd.to_numeric(df['Cogniçao (10-CS)'], errors='coerce').astype('Int64')
df['Cognição (MOCA)'] = pd.to_numeric(df['Cognição (MOCA)'], errors='coerce').astype('Int64')
df['Dinamômetro'] = pd.to_numeric(df['Dinamômetro'], errors='coerce').round().astype('Int64')

### Cálculo da velocidade no teste de 10 metros

A coluna `10 metros` registra o tempo em segundos. A velocidade é calculada em metros por segundo (m/s) pela distância de 10 metros dividida pelo tempo. Tempos ausentes ou não positivos geram velocidade ausente.

In [ ]:
df['10 metros'] = pd.to_numeric(df['10 metros'], errors='coerce')
df['Velocidade (m/s)'] = 10 / df['10 metros'].where(df['10 metros'] > 0)

In [ ]:
df['Sexo'] = df['Sexo'].astype('category')
df['Posição equilíbrio'] = df['Posição equilíbrio'].astype('category')
df['Panturrilha D'] = pd.to_numeric(df['Panturrilha D'], errors='coerce').astype('float')

In [ ]:
df.isnull().sum()

### Tratamento conservador das colunas de suporte e status

Padroniza caixa e espaços, converte células vazias e o valor "." em ausentes, sem agrupar descrições diferentes.

In [ ]:
colunas_texto_livre = [
    'Suporte social (terapêutico)',
    'Suporte social (domiciliar)',
    'Status funcional',
]

for coluna in colunas_texto_livre:
    df[coluna] = (
        df[coluna]
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.lower()
        .replace(['', '.'], pd.NA)
    )

colunas_suporte = [
    'Suporte social (terapêutico)',
    'Suporte social (domiciliar)',
]
df[colunas_suporte] = df[colunas_suporte].fillna('não informado')

In [ ]:
pd.DataFrame({
    'valores_distintos': df[colunas_texto_livre].nunique(dropna=True),
    'ausentes': df[colunas_texto_livre].isna().sum(),
})

In [ ]:
coluna_normalizada = df['Suporte social (terapêutico)'].astype(str).str.lower().str.strip()

valores_permitidos = ['não informado', 'independente']

condicao = ~coluna_normalizada.isin(valores_permitidos)

df.loc[condicao, 'Suporte social (terapêutico)'] = 'dependente'

print(df)

In [ ]:
coluna_normalizada = df['Suporte social (domiciliar)'].astype(str).str.lower().str.strip()

valores_permitidos = ['não informado', 'independente']

condicao = ~coluna_normalizada.isin(valores_permitidos)

df.loc[condicao, 'Suporte social (domiciliar)'] = 'dependente'

print(df)

In [ ]:
df['Queda último mês'].isnull().sum()

In [ ]:
df['Queda último mês'] = df['Queda último mês'].astype(str).str.strip()

df['Queda último mês'] = df['Queda último mês'].replace({
    '0': 'não',
    '2': 'sim',
    '': 'não informado',
    'nan': 'não informado',
    'NaN': 'não informado',
    'null': 'não informado',
    'None': 'não informado'
})

df['Queda último mês'] = df['Queda último mês'].fillna('não informado')
df['Queda último mês'] = df['Queda último mês'].astype('category')

In [ ]:
print(df['Queda último mês'].isna().sum())

print(df['Queda último mês'].value_counts(dropna=False))

In [ ]:
df['Nº quedas ao chão'] = df['Nº quedas ao chão'].astype(str).str.strip()

df['Nº quedas ao chão'] = df['Nº quedas ao chão'].replace({
    '': 'não informado',
    'nan': 'não informado',
    'NaN': 'não informado',
    'null': 'não informado',
    'None': 'não informado'
})

df['Nº quedas ao chão'] = df['Nº quedas ao chão'].fillna('não informado')

##### Tratamento de nulos

##### Padronizar os registros

In [ ]:
df['Clínica/Dx'] = df['Clínica/Dx'].astype('string').str.strip().str.upper()

# Dicionário de substituição dos diagnósticos
diagnosticos = {
    'AMP': 'AMP - Amputação',
    'AMP BILATERAL': 'AMP Bilateral - Amputação bilateral',
    'AMP LEA': 'AMP LEA - Amputação e Lesão Encefálica Adquirida',
    'AVC': 'AVC - Acidente Vascular Cerebral',
    'DNM': 'DNM - Doença Neuromuscular',
    'ELA': 'ELA - Esclerose Lateral Amiotrófica',
    'LEA': 'LEA - Lesão Encefálica Adquirida',
    'LM': 'LM - Lesão Medular',
    'PARKISON': 'Parkinson',
    'PARKINSON': 'Parkinson',
    'PC': 'PC - Paralisia Cerebral',
    'POLIO': 'Poliomielite',
    'POLIO?': 'Poliomielite',
    'MFC': 'MFC - Malformações Congênitas',
    'Mielo': 'Mielomeningocele',
    'EM': 'EM - Esclerose Múltipla',
    'LEA/LM': 'LEA/LM - Lesão Encefálica Adquirida e Lesão Medular'
}
df['Clínica/Dx'] = df['Clínica/Dx'].replace(diagnosticos)

### Tratativa da coluna status funcional

In [ ]:
# vendo quantidade de valores nulos na coluna 'Status funcional'
df['Status funcional'].isnull().sum()

# Substitui os vazios (e o 'nan' que virou texto) por "não informado"
df['Status funcional'] = df['Status funcional'].fillna('não informado')

# substitutindo o texto cadeira de rodas por cadeirante
df['Status funcional'] = df['Status funcional'].replace('cadeira de rodas', 'cadeirante')
df['Status funcional'] = df['Status funcional'].replace('deslocamento por cr', 'cadeirante')
df['Status funcional'] = df['Status funcional'].replace('marca comunitária', 'marcha comunitária')

#padronizando os textos para minúsculo
df['Status funcional'] = df['Status funcional'].str.lower()

#corrigindo os textos com erros de digitação
df['Status funcional'] = df['Status funcional'].replace({
    'marcha comunitária i': 'marcha comunitária independente',
    'cadeira de rodas ': 'cadeirante',
    'marhca comunitária com andador': 'marcha comunitária com andador',
    'marhca comunitária com bastão e supervisão': 'marcha comunitária com bastão e supervisão',
    'marcha comunitária i': 'marcha comunitária independente',
    
})

# removendo os '?' que ficaram no meio do texto
df['Status funcional'] = df['Status funcional'].str.replace('?', '', regex=False)

print(df['Status funcional'].value_counts())



In [ ]:
# Listas auxiliares para armazenar os resultados de cada linha
status_atualizado_list = []
supervisao_list = []
cadeirante_list = []
objetos_auxilio_list = []
nivel_locomocao_list = []

alvos_principais = [
    "deambuladora comunitária independente",
    "deambulador comunitário independente",
    "cadeirante longas distâncias",
    "deambulador comunitário",
    "deambulador domiciliar",
    "deambula independente",
    "marcha domiciliar",
    "marcha terapêutica",
    "marcha comunitária independente",
    "marcha comunitária",
    "marcha independente",
    "deambula",
    "cadeirante",
    "marcha"
]

termos_distancia = [
    "longas distâncias",
    "longa distância",
    "curtas distâncias",
    "curta distância"
]

In [ ]:
df['Status funcional'] = (
    df['Status funcional']
    .str.replace(r'(?i)\bdomicliar\b', 'domiciliar', regex=True)
    .str.strip()
)

In [ ]:
def processar_cadeirante(texto_lower):
    """Função responsável pela coluna 'Cadeirante'"""
    if 'não informado' in texto_lower:
        return 'não informado'
    
    if re.search(r'\bcad[eé]irante\b', texto_lower) or re.search(r'\bcadeira\b', texto_lower):
        return "sim"
    
    return "não"

def processar_supervisao(texto_str, texto_lower):
    """Função responsável pela coluna 'Supervisão'"""
    if 'não informado' in texto_lower:
        return 'não informado', texto_str
    
    # Se antes de supervisão tiver "sem", coloca "não" e remove as duas palavras
    if re.search(r'\bsem\s+sup[eé]rvis[aã]o\b', texto_lower):
        texto_limpo = re.sub(r'\bsem\s+sup[eé]rvis[aã]o\b', '', texto_str, flags=re.IGNORECASE)
        return "não", texto_limpo
    
    if re.search(r'\bcadeira\b', texto_lower):
        sup = "sim" if re.search(r'\bsup[eé]rvis[aã]o\b', texto_lower) else "não"
        return sup, texto_str
    
    if re.search(r'\bsup[eé]rvis[aã]o\b', texto_lower):
        texto_limpo = re.sub(r'\bsup[eé]rvis[aã]o\b', '', texto_str, flags=re.IGNORECASE)
        return "sim", texto_limpo
        
    return "não", texto_str

def processar_nivel_locomocao(texto_str, texto_lower):
    """Função responsável pela coluna 'Nivel de locomoção'"""
    if 'não informado' in texto_lower:
        return 'não informado', texto_str
        
    if re.search(r'\bcadeira\b', texto_lower):
        for td in termos_distancia:
            if td in texto_lower:
                return td, texto_str
        return "não especificado", texto_str
        
    for td in termos_distancia:
        pattern_td = re.compile(r'\b' + re.escape(td) + r'\b', re.IGNORECASE)
        if pattern_td.search(texto_str):
            texto_limpo = pattern_td.sub('', texto_str, count=1)
            return td, texto_limpo
            
    return "não especificado", texto_str

def processar_status_e_auxilio(texto_str, texto_lower):
    """Função responsável por atualizar 'Status funcional' e gerar 'Objetos de auxílio'"""
    if 'não informado' in texto_lower:
        return texto_str, 'não informado'
        
    if re.search(r'\bcadeira\b', texto_lower):
        return texto_str, "verificação"
        
    # Adiciona "marcha " se começar com domiciliar ou comunitária
    if texto_lower.strip().startswith('domiciliar') or texto_lower.strip().startswith('comunitária'):
        texto_str = "marcha " + texto_str
        texto_lower = "marcha " + texto_lower

    encontrado = None
    pattern_alvo = None
    
    for alvo in alvos_principais:
        # Usa \b para garantir que palavras isoladas não capturem partes de frases compostas indevidamente
        pattern_alvo = re.compile(r'\b' + re.escape(alvo) + r'\b', re.IGNORECASE)
        match = pattern_alvo.search(texto_str)
        if match:
            encontrado = match.group(0)
            break
            
    if encontrado:
        resto = pattern_alvo.sub('', texto_str, count=1)
        resto = re.sub(r'\s+', ' ', resto).strip()
        return encontrado, resto
    else:
        return texto_str.strip(), ""

In [ ]:
for texto in df['Status funcional']:
    if pd.isna(texto):
        status_atualizado_list.append(pd.NA)
        cadeirante_list.append(pd.NA)
        supervisao_list.append(pd.NA)
        objetos_auxilio_list.append(pd.NA)
        nivel_locomocao_list.append(pd.NA)
        continue
        
    texto_str = str(texto).strip()
    texto_lower = texto_str.lower()
    
    # Proteção caso o cabeçalho venha misturado nos dados
    if texto_str == 'Status funcional':
        status_atualizado_list.append(texto_str)
        cadeirante_list.append('Cadeirante')
        supervisao_list.append('Supervisão')
        objetos_auxilio_list.append('Objetos de auxílio')
        nivel_locomocao_list.append('Nivel de locomoção')
        continue

    # 1. Processa a coluna Cadeirante (sem remover as palavras do texto)
    cad_val = processar_cadeirante(texto_lower)

    # 2. Processa a supervisão (trata "sem supervisão" -> "não" e limpa o texto)
    sup_val, texto_str = processar_supervisao(texto_str, texto_lower)
    texto_lower = texto_str.lower()

    # 3. Processa o nível de locomoção e já limpa o texto
    niv_val, texto_str = processar_nivel_locomocao(texto_str, texto_lower)
    texto_lower = texto_str.lower()

    # 4. Processa o status final e os objetos de auxílio
    status_val, obj_val = processar_status_e_auxilio(texto_str, texto_lower)

    status_atualizado_list.append(status_val)
    cadeirante_list.append(cad_val)
    supervisao_list.append(sup_val)
    objetos_auxilio_list.append(obj_val)
    nivel_locomocao_list.append(niv_val)

# Atualiza o DataFrame existente preservando as outras colunas
df['Status funcional'] = status_atualizado_list

# Remove as colunas novas caso já existam para evitar duplicidade
for col in ['Cadeirante', 'Supervisão', 'Objetos de auxílio', 'Nivel de locomoção']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Insere as novas colunas na ordem correta logo após 'Status funcional'
idx = df.columns.get_loc('Status funcional')
df.insert(idx + 1, 'Cadeirante', cadeirante_list)
df.insert(idx + 2, 'Supervisão', supervisao_list)
df.insert(idx + 3, 'Objetos de auxílio', objetos_auxilio_list)
df.insert(idx + 4, 'Nivel de locomoção', nivel_locomocao_list)

print("Processamento executado com sucesso!")

In [ ]:
def limpar_com_excecoes(texto, palavras_manter, palavras_remover):
    if pd.isna(texto):
        return texto
    
    texto_str = str(texto)
    texto_lower = texto_str.lower()
    
    # Garante que os argumentos sejam listas (caso o usuário passe uma string solta)
    if isinstance(palavras_manter, str):
        palavras_manter = [palavras_manter]
    if isinstance(palavras_remover, str):
        palavras_remover = [palavras_remover]
    
    # Identifica quais termos para manter estão no texto
    encontradas_manter = [p for p in palavras_manter if p.lower() in texto_lower]
    
    # Verifica se QUALQUER termo da lista de remoção está presente no texto
    tem_remover = any(rem.lower() in texto_lower for rem in palavras_remover)
    
    if tem_remover:
        # Se encontrou termo(s) para manter, preserva apenas ele(s)
        if encontradas_manter:
            return ' e '.join(encontradas_manter)
        # Caso contrário, apaga tudo (deixa vazio)
        return ""
        
    return texto_str


In [ ]:
lista_palavras = ['muleta', 'bengala', 'bastão', 'aditamento']
lista_remover = ['apoio', 'simples', 'caminhada', 'total','tutor']

# Aplicando com lambda para passar a lista personalizada
df['Objetos de auxílio'] = df['Objetos de auxílio'].apply(
    lambda x: limpar_com_excecoes(x, lista_palavras, lista_remover)
)

In [ ]:
padrao = r'^(?:com|sem|e sem|e|simple|uso de)\b\s*|\s*\b(?:com|sem|e sem|e|simple|uso de)$'

# Aplicando a limpeza
df['Objetos de auxílio'] = (
    df['Objetos de auxílio']
    .astype(str)                                  # Garante que os dados são do tipo string
    .str.replace(padrao, '', case=False, regex=True) # Remove o padrão (case=False ignora maiúsculas/minúsculas)
    .str.strip()                                  # Remove espaços em branco desnecessários nas pontas
)
df['Objetos de auxílio'] = (
    df['Objetos de auxílio']
    .astype(str)
    .str.replace(padrao, '', case=False, regex=True)
    .str.strip()
)
df['Objetos de auxílio'] = (
    df['Objetos de auxílio']
    .str.replace(r'(?i)\bd\b', '', regex=True)
    .str.replace(r'(?i)\be\b', '-', regex=True)
    .str.replace(r'(?i)\bmuleta\b', 'muletas', regex=True)
    .str.replace(r'(?i)\bcanadense\b', 'canadenses', regex=True)
    .str.replace(r'(?i)\bcanandese\b', 'canadenses', regex=True)
    .str.replace(r'(?i)\baditamentos\b', 'aditamento', regex=True)
    .str.replace(r'(?i)\b longa\b', '', regex=True)
    .str.replace(r'[+/]+$', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df['Objetos de auxílio'] = (
    df['Objetos de auxílio']
    .str.replace(r'(?i)^\s*muletas\s*$', 'muletas axilares', regex=True)
)
df['Objetos de auxílio'] = (
    df['Objetos de auxílio']
    .str.replace(r'(?i)^\s*(canadenses\b)', r'muletas \1', regex=True)
    .str.replace(r'(?i)(\bmuletas\b)\s*$', r'\1 axilares', regex=True)
)

In [ ]:
df.loc[df['ID'] == 164, 'Objetos de auxílio'] = "andador - prótese"
df.loc[df['ID'] == 76, 'Objetos de auxílio'] = "prótese - andador"
df.loc[df['ID'] == 111, 'Objetos de auxílio'] = "andador"
df.loc[df['ID'] == 152, 'Objetos de auxílio'] = "andador"
df.loc[df['ID'] == 103, 'Objetos de auxílio'] = ""

df.loc[df['ID'] == 256, 'Supervisão'] = "sim"
df.loc[df['ID'] == 256, 'Objetos de auxílio'] = "muletas canadenses"

In [ ]:
df.loc[df['ID'] == 38, 'Status funcional'] = "cadeirante"
df.loc[df['ID'] == 38, 'Objetos de auxílio'] = ""

df.loc[df['ID'] == 43, 'Status funcional'] = "cadeirante"
df.loc[df['ID'] == 43, 'Objetos de auxílio'] = ""

df.loc[df['ID'] == 125, 'Status funcional'] = "cadeirante"
df.loc[df['ID'] == 125, 'Objetos de auxílio'] = ""

df.loc[df['ID'] == 160, 'Status funcional'] = "cadeirante"
df.loc[df['ID'] == 160, 'Objetos de auxílio'] = ""

df.loc[df['ID'] == 163, 'Status funcional'] = "cadeirante"
df.loc[df['ID'] == 163, 'Objetos de auxílio'] = ""

df.loc[df['ID'] == 142, 'Cadeirante'] = "sim"
df.loc[df['ID'] == 142, 'Objetos de auxílio'] = "muletas axilares"


In [ ]:
df[['Objetos de auxílio']]

In [ ]:
#visualizando os a colunas 'status funcional'
df[['Status funcional']]


### Salva as alterações diretamente no arquivo .csv

Facilita a visualização diretamente no arquivo

In [ ]:
df.to_csv('../dados/base_quedas_AACD_anonimizada.csv', index=False, encoding='utf-8-sig')
print("Arquivo salvo com sucesso!")